# The geometry of a track

The starter steers at a point three waypoints ahead and never looks further.
**Where exactly is the car relative to the track, and what is coming?**

Unit 1 timed the starter and found it capped at 20 m/s everywhere.
Look at what it knows while it drives: one number, the heading error to a waypoint a few metres ahead.
It does not know how far it is from the centreline, which way the track bends 50 m from now, or how tightly.
Every controller in this book needs those answers, so this unit builds the instruments that give them:

- **frames and heading**: the car's view of the world, `atan2`, and wrapping an angle to [−π, π);
- **projection**: where the car is along the track, found by projecting onto the waypoint polyline with a windowed search;
- **cross-track error**: the signed distance from the centreline, checked against racekit's own number to 1e-6 m;
- **curvature**: how sharply the track bends at every waypoint, from three points at a time.

A fair warning, before you expect a faster car: **this unit does not change the driving.**
On racekit's kinematic car the starter neither wobbles nor cuts corners.
Measured at every target speed from 20 to 35 m/s on all three tracks, its steer changes sign at most 0.08 times a second, it never strays more than 0.85 m from the centreline, and every race is clean.
Wobbling and corner-cutting are grip phenomena; they arrive with the dynamic car in Unit 6.

What this unit does is **measure**.
The most useful measurement turns out to be a failure: turn the starter's `STEER_GAIN` down, and watch it run wide in every corner and then hit the wall.

In [ ]:
import math
from inspect import getsource
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.runner import CONTROL_DT, load_submission, run_race, shim_installed
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")
UNIT_02 = Path("../../projects/racekit/agents/unit-02.py")
workdir = Path(mkdtemp(prefix="roar-unit-02-"))
deterministic = lambda: KinematicCar(jitter=0.0)

## 1. Frames, and the heading from `atan2`

Two frames matter.

- The **world frame**: `x`, `y` in metres, fixed to the track. The car's **yaw** ψ is its heading, measured counter-clockwise from the +x axis, in radians.
  The telemetry's `x`, `y`, `yaw` columns and the waypoints are all in this frame.
- The **car frame**: the origin at the car, `forward` along its heading and `left` to its left (the ROAR convention: x forward, y left).

A world point p seen from a car at c with yaw ψ is the offset p − c rotated by −ψ:

$$
\begin{pmatrix}\text{forward}\\ \text{left}\end{pmatrix}
=
\begin{pmatrix}\cos\psi & \sin\psi\\ -\sin\psi & \cos\psi\end{pmatrix}
\begin{pmatrix}p_x - c_x\\ p_y - c_y\end{pmatrix}.
$$

The **heading** of a vector (dx, dy) is `atan2(dy, dx)`, in (−π, π].
Never `atan(dy / dx)`: it loses the quadrant (a vector pointing back-left gets the heading of one pointing forward-right) and it divides by zero for a car heading due north.

In [ ]:
def to_car_frame(px, py, cx, cy, yaw):
    """The world point (px, py) seen from a car at (cx, cy) with heading yaw: (forward, left)."""
    dx, dy = px - cx, py - cy
    c, s = math.cos(yaw), math.sin(yaw)
    return c * dx + s * dy, -s * dx + c * dy


# A car at (100, 50) heading 120 degrees (up and to the left), and three points.
car_x, car_y, car_yaw = 100.0, 50.0, math.radians(120.0)
for name, (px, py) in {"ahead": (95.0, 58.66), "to its left": (91.34, 45.0), "behind": (105.0, 41.34)}.items():
    forward, left = to_car_frame(px, py, car_x, car_y, car_yaw)
    print(f"{name:>12}: forward {forward:6.2f} m, left {left:6.2f} m")

The starter's steering reads the same geometry in world coordinates: the heading error is the bearing of the target, `atan2(Δy, Δx)`, minus the yaw, wrapped (§2).
In the car frame the same angle is simply `atan2(left, forward)`.
The two must agree for every pose, which makes a good first test of `to_car_frame`.
The comparison wraps the difference, because −π and π are the same angle.

In [ ]:
def wrap_angle(angle):
    """Wrap an angle in radians to [-pi, pi)."""
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


worst = 0.0
for yaw in np.linspace(-math.pi, math.pi, 25):
    for tx, ty in [(3.0, 4.0), (-7.0, 1.0), (-2.0, -9.0), (5.0, -0.5)]:
        world_error = wrap_angle(math.atan2(ty - 1.0, tx - 2.0) - yaw)
        forward, left = to_car_frame(tx, ty, 2.0, 1.0, yaw)
        car_error = math.atan2(left, forward)
        worst = max(worst, abs(wrap_angle(world_error - car_error)))
print("largest disagreement over 100 poses:", worst, "rad")
assert worst < 1e-12

### The heading of a chord

Waypoints are points, so the track's heading between two of them is the heading of the **chord** joining them.
On a circular corner of radius R, take two points at angles θ₁ and θ₂ (counter-clockwise from the centre).
The chord is perpendicular to the radius that bisects it, so its heading is the tangent's heading at the **midpoint** angle, (θ₁ + θ₂)/2 + π/2, and not at either end.

In [ ]:
R, cx, cy = 18.0, 40.0, -25.0
worst = 0.0
for theta1, theta2 in [(0.0, 0.11), (1.0, 1.3), (2.9, 3.4), (-2.0, -1.2), (4.0, 4.111)]:
    ax_, ay_ = cx + R * math.cos(theta1), cy + R * math.sin(theta1)
    bx_, by_ = cx + R * math.cos(theta2), cy + R * math.sin(theta2)
    chord = math.atan2(by_ - ay_, bx_ - ax_)
    tangent_at_midpoint = (theta1 + theta2) / 2.0 + math.pi / 2.0  # counter-clockwise travel
    worst = max(worst, abs(wrap_angle(chord - tangent_at_midpoint)))
print("chord heading vs tangent at the midpoint angle:", worst, "rad")
assert worst < 1e-12

half_step = 2.0 / 18.0 / 2.0  # waypoints 2 m apart on the 18 m hairpin of test-circuit
print(f"so at a waypoint the chord's heading is off the tangent by half a step: {half_step:.4f} rad = {math.degrees(half_step):.1f} degrees")

That half-step is a real error in the tightest corner of `test-circuit` (3.2°), and it is why the yaw stored with each waypoint is not exactly the heading of the next chord.
Keep it in mind whenever you compare a heading from waypoints with the car's yaw.

## 2. Wrapping

Two headings can be equal and differ by 2π.
Any time you **subtract** angles, wrap the result back to one canonical interval.
This book uses [−π, π):

$$\operatorname{wrap}(a) = \big((a + \pi) \bmod 2\pi\big) - \pi .$$

Python's `%` with a positive divisor always returns a value in [0, 2π), even for a negative `a`.
That is the property the formula needs.
C and C++ programmers, beware: `fmod` keeps the sign of `a`, so the same formula in C++ is wrong for negative angles.

The edge cases are where wrapping code breaks, so pin them: both π and −π map to −π (the interval is closed at −π), so do ±3π, 0 stays 0, and wrapping twice changes nothing.

In [ ]:
cases = {"pi": math.pi, "-pi": -math.pi, "3 pi": 3 * math.pi, "-3 pi": -3 * math.pi, "0": 0.0}
for name, angle in cases.items():
    print(f"wrap({name:>5}) = {wrap_angle(angle): .15f}")

assert wrap_angle(math.pi) == -math.pi and wrap_angle(-math.pi) == -math.pi
assert wrap_angle(3 * math.pi) == -math.pi and wrap_angle(-3 * math.pi) == -math.pi
assert wrap_angle(0.0) == 0.0
for angle in np.linspace(-20.0, 20.0, 401):
    once = wrap_angle(angle)
    assert -math.pi <= once < math.pi and wrap_angle(once) == once  # in range, and idempotent
    assert abs(math.sin(once) - math.sin(angle)) < 1e-12 and abs(math.cos(once) - math.cos(angle)) < 1e-12

What an unwrapped heading error does to the starter: the car heads at 179°, and its target sits at a bearing of −179°, just 2° to its left.
The starter's steer is `-STEER_GAIN / sqrt(v) · error / π`, clipped to [−1, 1] (positive steers right).

In [ ]:
yaw, bearing, v = math.radians(179.0), math.radians(-179.0), 20.0
naive = bearing - yaw
wrapped = wrap_angle(bearing - yaw)


def starter_steer(error, speed=v, gain=8.0):
    return float(np.clip(-gain / math.sqrt(speed) * error / math.pi, -1.0, 1.0))


print(f"naive error {math.degrees(naive):8.2f} deg -> steer {starter_steer(naive):+.3f}")
print(f"wrapped     {math.degrees(wrapped):8.2f} deg -> steer {starter_steer(wrapped):+.3f}")
assert starter_steer(naive) == 1.0 and starter_steer(wrapped) < 0.0

Unwrapped, a target 2° to the left produces **full right lock**.
The starter wraps; a controller you write must too, and so must every analysis that differences the `yaw` column (Exercise 2 shows what the raw column does on the `oval`).

## 3. Projection onto a segment, then onto the track

The centreline is a closed **polyline**: segment i runs from waypoint i to waypoint i + 1, and the last one wraps to waypoint 0.

**Onto one segment.** With a → b the segment, d = b − a and e = p − a, the foot of the perpendicular from p is a + t·d with

$$ t = \frac{e \cdot d}{|d|^2}. $$

t < 0 means p is before a, t > 1 means it is past b.
For a **segment** the foot is clamped to the segment: t is clipped to [0, 1], and the distance is measured to that clamped foot.

**Onto the polyline.** The segment with the smallest clamped-foot distance wins.
Searching all n segments every tick costs O(n): 2857 segments on `monza-like`, 20 times a second.
The car moves about 1 m per tick, so the right segment is always near the last one.
A **windowed search** examines only the segments `hint − 8 … hint + 8`, indexed **cyclically** (`k % n`, so the window runs across the start line), and carries the winner forward as the next tick's `hint`.

The rules, pinned so that two implementations agree exactly:

- the window is `hint − 8 … hint + 8`, and `hint` starts at 0;
- the smallest **squared** clamped-foot distance wins (the square root changes nothing in the comparison);
- ties go to the first segment examined, because the comparison is a strict `<`.

In [ ]:
def project_onto_segment(px, py, ax, ay, bx, by):
    """(t clamped to [0, 1], squared distance from p to the clamped foot) for the segment a -> b."""
    dx, dy = bx - ax, by - ay
    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0.0, min(t, 1.0))
    fx, fy = px - (ax + t * dx), py - (ay + t * dy)
    return t, fx * fx + fy * fy


def nearest_segment(px, py, xs, ys, hint, reach=8):
    """The windowed search: the segment in hint - reach .. hint + reach (cyclic) whose clamped foot is nearest."""
    n = len(xs)
    best_d2, best = math.inf, hint % n
    for k in range(hint - reach, hint + reach + 1):
        i = k % n
        j = (i + 1) % n
        _, d2 = project_onto_segment(px, py, xs[i], ys[i], xs[j], ys[j])
        if d2 < best_d2:
            best_d2, best = d2, i
    return best


print(project_onto_segment(3.0, 4.0, 0.0, 0.0, 10.0, 0.0))    # foot inside: t = 0.3, distance 4
print(project_onto_segment(13.0, 4.0, 0.0, 0.0, 10.0, 0.0))   # past b: t clamped to 1, distance 5

Now a real track.
`load_track(name)` gives the waypoints as arrays `track.x`, `track.y`; plain Python floats make the per-row loops several times faster than NumPy scalars, so we convert once.
A point 1.5 m to the left of the track just after waypoint 2855 of `monza-like` is found from `hint = 2` across the start line, and it agrees with a brute-force search over every segment.

In [ ]:
monza = load_track("monza-like")
mx, my = [float(value) for value in monza.x], [float(value) for value in monza.y]
n_monza = len(mx)
print(monza, "| first and last waypoint:", (round(mx[0], 1), round(my[0], 1)), (round(mx[-1], 1), round(my[-1], 1)))

# 30 % of the way along segment 2855, then 1.5 m to its left
i = 2855
dx, dy = mx[i + 1] - mx[i], my[i + 1] - my[i]
length = math.hypot(dx, dy)
px = mx[i] + 0.3 * dx - 1.5 * dy / length
py = my[i] + 0.3 * dy + 1.5 * dx / length

windowed = nearest_segment(px, py, mx, my, hint=2)
brute = min(range(n_monza), key=lambda k: project_onto_segment(px, py, mx[k], my[k], mx[(k + 1) % n_monza], my[(k + 1) % n_monza])[1])
print("windowed search from hint 2:", windowed, "| brute force over all", n_monza, "segments:", brute)
assert windowed == brute == 2855

On these tracks a brute-force search would give the same answer for any point on the track: the closest that two far-apart parts of `test-circuit` come is about 37 m (the two legs of its hairpin), more than six half-widths.
The window is there for **cost**, and for tracks where two legs do run close.
Its failure mode is a **stale hint**: a window around the wrong place returns the nearest segment *in the window*, with no warning (Exercise 3).
racekit's race loop carries its hint exactly like this, from 0, every tick.

This is not the runner's **progress** rule.
Progress (Unit 1) projects the next 15 **waypoints** onto the car's **latest motion segment** and only moves forward, because it answers a different question: how far along the race the car has got, monotonically.
The instrument here answers *where is the car now*, and it may move backwards.

## 4. The cross-track error

The **cross-track error** is the car's signed distance from the centreline: positive to the **left** of the direction of travel, as in racekit's `offset` column.
For the chosen segment, with d = b − a and e = p − a, it is the scalar cross product divided by |d|:

$$ \text{offset} = \frac{d_x e_y - d_y e_x}{|d|} = |e|\,\sin\varphi, $$

where φ is the angle from d to e: positive when p is to the left of a → b.
It is the signed distance from p to the segment's **line**.

Two practical notes.

- Write the scalar, not `np.cross`: NumPy 2 refuses 2-vectors in `np.cross`, and the submission must run on Python 3.8 with plain floats anyway.
- racekit measures from the foot, e = p − foot, instead of from a. That is the same number: the two differ by a multiple of d, and the cross product of d with itself is zero.

In [ ]:
def signed_offset(px, py, ax, ay, bx, by):
    """Signed distance from p to the line through a -> b, left positive."""
    dx, dy = bx - ax, by - ay
    ex, ey = px - ax, py - ay
    return (dx * ey - dy * ex) / math.sqrt(dx * dx + dy * dy)


def cross_track(px, py, xs, ys, hint, reach=8):
    """(segment index, signed cross-track error): the windowed search, then the offset to the chosen segment."""
    i = nearest_segment(px, py, xs, ys, hint, reach)
    j = (i + 1) % len(xs)
    return i, signed_offset(px, py, xs[i], ys[i], xs[j], ys[j])


def track_position(telemetry, xs, ys):
    """Run cross_track over every row, carrying the hint from 0: arrays (segment, offset).

    A held row (the respawn hold, at the start pose) restarts the search from hint 0, as racekit does."""
    hint, segments, offsets = 0, [], []
    for x, y, held in zip(telemetry["x"], telemetry["y"], telemetry["held"]):
        hint, offset = cross_track(float(x), float(y), xs, ys, 0 if held else hint)
        segments += [hint]
        offsets += [offset]
    return np.array(segments), np.array(offsets)

### Checked on a circle

Make a **counter-clockwise** circle of radius R = 50 m as a 60-point polyline.
Left of a counter-clockwise path is the **inside**, so a point at radius r from the centre should have offset R − r.
A polyline is not a circle: each chord sits inside the arc by up to the **sagitta**, R(1 − cos(Δθ/2)) with Δθ = 2π/60, so that is the tolerance.
The same circle driven **clockwise** puts the inside on the right, and every sign flips.

In [ ]:
R, N = 50.0, 60
angles = np.linspace(0.0, 2.0 * math.pi, N + 1)[:-1]
ccw_x, ccw_y = [R * math.cos(a) for a in angles], [R * math.sin(a) for a in angles]
cw_x, cw_y = ccw_x[::-1], ccw_y[::-1]
sagitta = R * (1.0 - math.cos(math.pi / N))

worst_ccw = worst_cw = 0.0
for r in [44.0, 49.0, 50.0, 51.5, 56.0]:
    for phi in np.linspace(0.05, 6.2, 37):
        px, py = r * math.cos(phi), r * math.sin(phi)
        start = int(np.argmin(np.hypot(np.array(ccw_x) - px, np.array(ccw_y) - py)))
        _, ccw_offset = cross_track(px, py, ccw_x, ccw_y, start)
        _, cw_offset = cross_track(px, py, cw_x, cw_y, N - 1 - start)
        worst_ccw = max(worst_ccw, abs(ccw_offset - (R - r)))
        worst_cw = max(worst_cw, abs(cw_offset + (R - r)))
print(f"sagitta bound {sagitta:.4f} m | worst |offset - (R - r)| counter-clockwise {worst_ccw:.4f} m, clockwise (sign flipped) {worst_cw:.4f} m")
assert worst_ccw <= sagitta and worst_cw <= sagitta

### Checked against racekit, on every row

racekit computes `offset` with exactly these rules (its `Track.nearest`, from hint 0, carried each tick).
So your instrument should reproduce the column on **every row** of a deterministic race, not just on average.
This is the full `monza-like` race at jitter 0: 17651 rows.

In [ ]:
monza_race = run_race(STARTER, "monza-like", car=deterministic())
monza_tel = monza_race.telemetry
print(monza_race)
assert monza_race.finished and monza_race.collisions == 0 and abs(monza_race.total_time - 882.55) < 1e-9

monza_segments, monza_offsets = track_position(monza_tel, mx, my)
difference = np.abs(monza_offsets - monza_tel["offset"])
print(len(monza_tel), "rows; largest |yours - racekit's|:", float(np.max(difference)), "m")
assert np.max(difference) < 1e-6

Agreement to about 1e-13 m: float rounding.

Here is why the formula is pinned.
A natural alternative is "the distance to the clamped foot, with the sign of the cross product".
It is the same number whenever the foot lies inside the segment.
When the car is just past a waypoint on the **outside** of a bend, the nearest segment's foot is clamped to its end, and the distance to an endpoint is longer than the distance to the line.

In [ ]:
naive = []
clamped = []
for row, segment in enumerate(monza_segments):
    x, y = float(monza_tel["x"][row]), float(monza_tel["y"][row])
    j = (segment + 1) % n_monza
    t, d2 = project_onto_segment(x, y, mx[segment], my[segment], mx[j], my[j])
    naive += [float(np.sign(monza_offsets[row])) * math.sqrt(d2)]
    clamped += [t == 0.0 or t == 1.0]
naive, clamped = np.array(naive), np.array(clamped)

disagree = np.abs(naive - monza_tel["offset"]) > 1e-9
print("rows where the foot is clamped:", int(np.sum(clamped)))
print("rows where sign x foot-distance disagrees with racekit:", int(np.sum(disagree)), "| of them clamped:", int(np.sum(clamped[disagree])))
print("largest disagreement:", float(np.max(np.abs(naive - monza_tel["offset"]))), "m")
assert int(np.sum(disagree)) == 3 and int(np.sum(clamped[disagree])) == 3
assert abs(np.max(np.abs(naive - monza_tel["offset"])) - 4.9e-4) < 0.05e-4

Three rows out of 17651, and half a millimetre at worst.
Harmless for driving, but it means the naive instrument is not the one the judge uses.
The judge is `|offset| > half-width`: that is racekit's major collision (Unit 1).
An instrument that disagrees with the judge, even rarely, cannot be trusted exactly where it matters, at the edge.

With the right instrument, the **margin** is the half-width minus |offset|: how close the starter ever comes to the wall.

In [ ]:
margin = monza.half_width[monza_segments] - np.abs(monza_offsets)
print(f"max |offset| {np.max(np.abs(monza_offsets)):.2f} m, RMS {np.sqrt(np.mean(monza_offsets ** 2)):.3f} m; smallest margin to the edge {np.min(margin):.2f} m")
assert abs(np.max(np.abs(monza_offsets)) - 0.37) < 0.005

At 20 m/s the starter never comes within 5 m of the edge of `monza-like`.
That is the deterministic car; Unit 1's jittered race reached 0.61 m.

## 5. Curvature, from three points

How sharply does the track bend at a waypoint?
Take the waypoint and its two neighbours, and find the circle through all three.
Its radius R is the local radius of the corner, and the **curvature** is κ = 1/R (in 1/m; a straight has κ = 0).

For a triangle with sides a, b, c and area A, the law of sines gives a = 2R sin α, and A = ½ bc sin α.
Eliminating sin α gives the circumradius R = abc / (4A), so

$$ \kappa = \frac{1}{R} = \frac{4A}{abc}. $$

This is the **Menger curvature**.
Use the **signed** area, A = ½ (b − a) × (c − a) with the same scalar cross product as §4, and κ is **positive for a left turn** and negative for a right turn, matching the offset's convention.
Collinear points give A = 0, so a straight gives κ = 0 with no special case (only coincident points divide by zero).

In [ ]:
def menger_curvature(ax, ay, bx, by, cx, cy):
    """Signed curvature of the circle through a, b, c: 4 * area / (product of the sides), left turns positive."""
    cross = (bx - ax) * (cy - ay) - (by - ay) * (cx - ax)  # twice the signed area
    sides = math.hypot(bx - ax, by - ay) * math.hypot(cx - bx, cy - by) * math.hypot(ax - cx, ay - cy)
    return 2.0 * cross / sides


worst = 0.0
for R in [18.0, 25.0, 120.0, 1000.0]:
    for start, step in [(0.0, 2.0 / R), (1.0, 0.4), (-2.5, 1.1)]:
        ccw = [(7.0 + R * math.cos(start + k * step), -3.0 + R * math.sin(start + k * step)) for k in range(3)]
        left_turn = menger_curvature(*ccw[0], *ccw[1], *ccw[2])
        right_turn = menger_curvature(*ccw[2], *ccw[1], *ccw[0])  # the same points driven clockwise
        worst = max(worst, abs(left_turn - 1.0 / R), abs(right_turn + 1.0 / R))
print("largest |kappa - (+-1/R)| over 12 circles, both directions:", worst)
assert worst < 1e-9

### The curvature profile

Apply it at every waypoint, cyclically (waypoint 0's neighbours are the last waypoint and waypoint 1).
racekit's track generator stores the curvature it built each track with, `track.curvature`, so the profile can be checked against the truth.
Two more checks come free: the smallest radius on each track, and the **total turning** Σ κ·ds over one lap, which must be ±2π for a closed loop: +2π counter-clockwise, −2π clockwise.

In [ ]:
def curvature_profile(xs, ys):
    """The signed Menger curvature at every waypoint of a closed track."""
    n = len(xs)
    return np.array([menger_curvature(xs[i - 1], ys[i - 1], xs[i], ys[i], xs[(i + 1) % n], ys[(i + 1) % n]) for i in range(n)])


profiles = {}
for name, r_min in [("oval", 120.00), ("test-circuit", 18.00), ("monza-like", 25.00)]:
    track = load_track(name)
    xs, ys = [float(value) for value in track.x], [float(value) for value in track.y]
    kappa = curvature_profile(xs, ys)
    profiles[name] = (track, kappa)
    step_lengths = np.hypot(np.diff(track.x, append=track.x[0]), np.diff(track.y, append=track.y[0]))
    turning = float(np.sum(kappa * step_lengths))
    gap = float(np.max(np.abs(kappa - track.curvature)))
    print(f"{name:>12}: max |Menger - generator| {gap:.1e} 1/m | R_min {1.0 / np.max(np.abs(kappa)):6.2f} m | "
          f"total turning {turning / math.pi:+.4f} pi | left-turn share of curved waypoints {np.mean(kappa[np.abs(kappa) > 1e-3] > 0):.2f}")
    assert gap < 1e-3
    assert abs(1.0 / np.max(np.abs(kappa)) - r_min) < 0.005
    assert abs(abs(turning) - 2.0 * math.pi) < 0.01

Three readings.

- Menger agrees with the generator to better than 1e-3 1/m.
  The largest gaps sit where the curvature **changes**, at the entry of a corner: a circle through three points cannot represent a curvature that varies between them.
- R_min is 120 m on the `oval`, 18 m on `test-circuit` and 25 m on `monza-like`.
- `oval` and `test-circuit` run counter-clockwise (+2π, mostly left turns), but **`monza-like` runs clockwise** (−2π): most of its corners are right-handers, like the real Monza.
  Never assume a sign; measure it.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
for ax, name in zip(axes, ["test-circuit", "monza-like"]):
    track, kappa = profiles[name]
    ax.plot(track.s, kappa, linewidth=1.0)
    ax.axhline(0.0, color="0.5", linewidth=0.8)
    r_min = 1.0 / np.max(np.abs(kappa))
    ax.axhline(1.0 / r_min, color="tab:red", linestyle="--", linewidth=0.8, label=f"±1/R_min, R_min = {r_min:.0f} m")
    ax.axhline(-1.0 / r_min, color="tab:red", linestyle="--", linewidth=0.8)
    ax.set_title(f"{name}: signed curvature along the lap (left turns positive)")
    ax.set_xlabel("s, distance along the lap (m)")
    ax.set_ylabel("kappa (1/m)")
    ax.legend()
fig.tight_layout()
plt.show()

Read a corner off this plot: its sign is its direction, its height is 1/R, and its width is its length.
The profile is a list of "what is coming": at any waypoint, the next 25 entries cover the next 50 m.

## 6. Measuring the starter

Now point the instruments at the starter on `test-circuit` at its own speed, 20 m/s.
For each telemetry row, `track_position` gives the segment the car is on, so `kappa[segment]` is the curvature under the car.
The question: when the car is in a corner, which side of the centreline is it on?

In [ ]:
tc_track, tc_kappa = profiles["test-circuit"]
tc_x, tc_y = [float(value) for value in tc_track.x], [float(value) for value in tc_track.y]


def sweep_race(path):
    race = run_race(path, "test-circuit", car=deterministic(), max_seconds=1500)
    segments, offsets = track_position(race.telemetry, tc_x, tc_y)
    assert np.max(np.abs(offsets - race.telemetry["offset"])) < 1e-6  # the instrument still matches racekit
    driving = race.telemetry["held"] == 0  # drop the respawn-hold rows (the car is parked at the start)
    return race, race.telemetry[driving], segments[driving], offsets[driving]


gain_8, rows_8, segments_8, offsets_8 = sweep_race(STARTER)
kappa_8 = tc_kappa[segments_8]
in_corner = np.abs(kappa_8) > 0.01  # tighter than R = 100 m
outside = offsets_8[in_corner] * kappa_8[in_corner] < 0  # offset and curvature of opposite sign
print(gain_8)
print(f"rows in a corner: {int(np.sum(in_corner))}; on the outside of the corner: {np.mean(outside):.3f}")
print(f"mean |offset| in corners {np.mean(np.abs(offsets_8[in_corner])):.3f} m, on straights (|kappa| < 0.001) {np.mean(np.abs(offsets_8[np.abs(kappa_8) < 0.001])):.4f} m")
assert np.sum(~outside) == 0

**Every** corner row is on the outside.
The starter does not cut corners; it runs **wide**, by about 0.18 m on average and 0.51 m at most, while on the straights it sits within a millimetre of the line.

Why wide?
Its steering is proportional to the heading error, so holding a steady steer through a corner needs a steady, nonzero heading error toward the target.
On a curve, the only way to keep the target off the car's nose is to sit off the line, on the outside.
A weaker gain needs a bigger error for the same steer, so it should run wider.
That is a prediction we can test with `write_variant`, one constant at a time.

In [ ]:
SWEEP = {  # gain: (finished, collisions, total_time, max |offset|), at 20 m/s on test-circuit, jitter 0
    8.0: (True, 0, 412.75, 0.51),
    4.0: (True, 0, 413.55, 1.76),
    2.0: (True, 0, 415.35, 4.40),
    1.0: (False, 16, 1500.00, 5.64),
}
sweep = {8.0: (gain_8, rows_8, segments_8, offsets_8)}
for gain in [4.0, 2.0, 1.0]:
    sweep[gain] = sweep_race(write_variant(STARTER, workdir / f"gain_{gain:g}.py", {"STEER_GAIN": str(gain)}))

print(" gain | finished | collisions | time (s) | max |offset| (m) | where (waypoint, R)")
for gain, (race, rows, segments, offsets) in sweep.items():
    worst_row = int(np.argmax(np.abs(offsets)))
    where = segments[worst_row]
    print(f" {gain:4.0f} | {str(race.finished):>8} | {race.collisions:10d} | {race.total_time:8.2f} | {np.max(np.abs(offsets)):16.2f} | {where}, {1.0 / abs(tc_kappa[where]):.1f} m")
    finished, collisions, total_time, max_offset = SWEEP[gain]
    assert race.finished == finished and race.collisions == collisions
    assert abs(race.total_time - total_time) < 1e-9
    assert abs(np.max(np.abs(rows["offset"])) - max_offset) <= 0.005

The prediction holds, and then some:

| `STEER_GAIN` | outcome | max \|offset\| |
|---|---|---|
| 8 (the starter) | 412.75 s, clean | 0.51 m |
| 4 | 413.55 s, clean | 1.76 m |
| 2 | 415.35 s, clean | 4.40 m |
| 1 | **does not finish** in 1500 s: 16 collisions | 5.64 m |

Each halving of the gain multiplies the peak excursion by 2.5 to 3.5, and the peaks all sit in one place: the **R = 18 m hairpin** around waypoints 820–841, the tightest corner on the track (gain 2 peaks just after it, where the radius opens to 24 m).
`test-circuit` is 5.5 m wide there on each side, so at gain 1 the excursion the controller needs is wider than the road: it hits the wall in the hairpin, respawns, and does it again, 16 times.
On `monza-like` the same sweep gives 0.37 / 1.27 / 3.63 m, and gain 1 does not finish there either (22 collisions).

The plot puts the curvature under the car next to the offset, for gain 8 and gain 2: every corner leaves an offset of the opposite sign, scaled by how tight it is.

In [ ]:
fig, (ax_kappa, ax_offset) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
lap = rows_8["t"] <= 140.0  # about the first lap
ax_kappa.plot(rows_8["t"][lap], kappa_8[lap], linewidth=1.0)
ax_kappa.set_ylabel("kappa under the car (1/m)")
ax_kappa.set_title("test-circuit, 20 m/s: the curvature under the car, and the offset it causes")
for gain in [8.0, 2.0]:
    race, rows, segments, offsets = sweep[gain]
    keep = rows["t"] <= 140.0
    ax_offset.plot(rows["t"][keep], offsets[keep], linewidth=1.0, label=f"STEER_GAIN {gain:g}")
ax_offset.axhline(5.5, color="0.35", linewidth=1.0, label="edge (half-width 5.5 m)")
ax_offset.axhline(-5.5, color="0.35", linewidth=1.0)
ax_offset.set_xlabel("t (s)")
ax_offset.set_ylabel("offset (m, left +)")
ax_offset.legend()
fig.tight_layout()
plt.show()

## 7. A sharper instrument is not a better controller

The starter's notion of *where am I* is crude.
`advance_index` takes the **first waypoint within 3 m**, searching forward.
Waypoints are 2 m apart, so that waypoint is usually one **behind** the car.
The windowed nearest-waypoint search of §3 is sharper: it picks the waypoint nearest the car.

Measure both on the starter's own `test-circuit` race, by replaying each index rule over the telemetry and reading the index waypoint, and the target three past it, in the car frame.

In [ ]:
def reached_radius_index(index, x, y, xs, ys):
    for step in range(len(xs)):  # the starter's rule
        k = (index + step) % len(xs)
        if math.hypot(x - xs[k], y - ys[k]) < 3.0:
            return k
    return index


def windowed_nearest_index(index, x, y, xs, ys, window=20):
    best, best_d = index, math.inf  # the nearest of the next `window` waypoints
    for step in range(window):
        k = (index + step) % len(xs)
        d = math.hypot(x - xs[k], y - ys[k])
        if d < best_d:
            best, best_d = k, d
    return best


n_tc = len(tc_x)
starter_index = windowed_index = 0
index_ahead = {"reached-radius": [], "windowed": []}
target_ahead = {"reached-radius": [], "windowed": []}
for x, y, yaw in zip(rows_8["x"], rows_8["y"], rows_8["yaw"]):
    x, y, yaw = float(x), float(y), float(yaw)
    starter_index = reached_radius_index(starter_index, x, y, tc_x, tc_y)
    windowed_index = windowed_nearest_index(windowed_index, x, y, tc_x, tc_y)
    for rule, k in [("reached-radius", starter_index), ("windowed", windowed_index)]:
        index_ahead[rule] += [to_car_frame(tc_x[k], tc_y[k], x, y, yaw)[0]]
        target = (k + 3) % n_tc
        target_ahead[rule] += [to_car_frame(tc_x[target], tc_y[target], x, y, yaw)[0]]
for rule in index_ahead:
    print(f"{rule:>14}: index waypoint {np.mean(index_ahead[rule]):+.2f} m ahead of the car; target {np.mean(target_ahead[rule]):.2f} m ahead")
assert abs(np.mean(index_ahead["reached-radius"]) + 2.0) < 0.05 and abs(np.mean(index_ahead["windowed"])) < 0.05

The starter's index sits 2 m behind the car, so its target is about 4 m ahead.
The windowed index sits at the car, so with the same `LOOKAHEAD_WAYPOINTS = 3` the target moves to about 6 m ahead.

So swap the sharper instrument into the starter and change nothing else.
The variant is the starter's file plus a subclass that replaces `advance_index`, written into `workdir`; `write_variant` then sets its gain.

In [ ]:
WINDOWED = '''

SEARCH_WINDOW = 20  # waypoints searched ahead of the current index


_Starter = RoarCompetitionSolution


class RoarCompetitionSolution(_Starter):
    """The starter, with a windowed nearest-waypoint search in place of the reached-radius rule."""

    def advance_index(self, location):
        count = len(self.maneuverable_waypoints)
        best, best_d = self.current_index, float("inf")
        for step in range(SEARCH_WINDOW):
            index = (self.current_index + step) % count
            d = float(np.linalg.norm(location[:2] - self.maneuverable_waypoints[index].location[:2]))
            if d < best_d:
                best, best_d = index, d
        return best
'''

windowed_path = workdir / "windowed.py"
with open(STARTER) as source, open(windowed_path, "w") as target:
    target.write(source.read() + WINDOWED)

print(" gain | starter                | windowed search")
for gain, cap in [(8.0, 1500), (4.0, 1500), (2.0, 450)]:
    variant = write_variant(windowed_path, workdir / f"windowed_{gain:g}.py", {"STEER_GAIN": str(gain)})
    race = run_race(variant, "test-circuit", car=deterministic(), max_seconds=cap)
    rows = race.telemetry[race.telemetry["held"] == 0]
    starter_race = sweep[gain][0]
    print(f" {gain:4.0f} | {starter_race.total_time:7.2f} s, {starter_race.collisions:2d} crashes | "
          f"{'finished' if race.finished else 'DNF at'} {race.total_time:7.2f} s, {race.collisions:2d} crashes, max |offset| {np.max(np.abs(rows['offset'])):.2f} m")
    if gain == 2.0:
        assert not race.finished and race.collisions == 8  # in the first 450 s the starter needed for 3 clean laps
    else:
        assert race.finished and race.collisions == 0

At the starter's gain of 8, the sharper instrument is a wash: 412.70 s against 412.75 s is **one 0.05 s tick**, far inside the 0.3 s run-to-run noise of Unit 1, and the peak offset drops to 0.40 m.
At gain 4 it is slightly worse (413.75 s, 1.93 m).
At gain 2, where the starter finished clean, the "better" instrument **crashes 8 times in 450 s** (over a full 1500 s it would be 32 crashes, and no finish).

The instrument did not make the car worse; it **retuned the controller**.
For a target D metres ahead and a car y metres off the line, the heading error contains a term of about −y/D.
Moving the target from 4 m to 6 m ahead cuts that correction by a third: the windowed car at gain 2 corrects its offset like the starter at a gain of about 2 × 4/6 ≈ 1.3, between the starter's clean gain 2 and its crashing gain 1.

The lesson is general: **a controller is tuned to its instrument.**
Changing the instrument changes the controller, even when the new instrument is more accurate.
So this book's Unit 2 submission keeps the starter's driving, and only *adds* the instruments.
Units 4 and 5 build controllers designed around the cross-track error and the curvature ahead, and tune them with the noise-aware rule.

## 8. The submission, and the bridge to Unit 3

`agents/unit-02.py` is the starter with the instruments built in.
Every tick, after the starter's own index update, `measure()` computes the cross-track error with the windowed projection, **anchored at `self.current_index`** (the reached-radius index the controller already uses, so the two can never drift apart, and a respawn needs no special case), and the sharpest curvature in the next 25 waypoints (about 50 m).
Both are stored on `self`; neither touches the controls.

In [ ]:
with shim_installed():
    Unit02, _ = load_submission(UNIT_02)
print(getsource(Unit02.measure))

report = check_submission(UNIT_02)
print(report)
assert report.ok

A recording subclass reads the instrument after every step.
`step()` sees the car where the previous tick left it, so its reading on step k matches the telemetry's offset on row k − 1.

In [ ]:
recorded = {}


class Recorder(Unit02):
    async def initialize(self):
        await super().initialize()
        recorded["cross_track"], recorded["curvature_ahead"] = [], []

    async def step(self):
        await super().step()
        recorded["cross_track"] += [self.cross_track_error]
        recorded["curvature_ahead"] += [self.curvature_ahead]


oval_race = run_race(Recorder, "oval", car=deterministic())
cross_track_log = np.array(recorded["cross_track"])
gap = np.max(np.abs(cross_track_log[1:] - oval_race.telemetry["offset"][:-1]))
print(oval_race, "| the starter's time is 223.50 s")
print(f"in-car cross-track error vs racekit's offset: largest gap {gap:.1e} m")
print(f"sharpest curvature ever seen ahead: {np.max(np.abs(recorded['curvature_ahead'])):.5f} 1/m (R = {1.0 / np.max(np.abs(recorded['curvature_ahead'])):.1f} m)")
assert abs(oval_race.total_time - 223.50) < 1e-9 and oval_race.collisions == 0
assert gap < 1e-4

The submission races exactly like the starter (223.50 s), and its in-car cross-track error matches racekit's offset to about 2e-5 m.
That is not 1e-13 this time, because the sensors deliver `float32` positions, as the real interface does, and a `float32` coordinate a few hundred metres from the origin is only good to about 1e-5 m.

So the car now knows where it is, and what is coming: in the `test-circuit` hairpin, a curvature of 1/18 m⁻¹.
What it does not know is **what steer that corner needs**, or where a given steer will put the car one second from now.
That takes a model of the car itself: the kinematic bicycle, its throttle and drag, and how to integrate them.
That is Unit 3.